# 🎯 Porównanie 5 podejść do klasyfikacji zgłoszeń IT

Od ręcznej klasyfikacji, przez ML, po LLM i RAG — porównujemy skuteczność różnych metod na **tych samych 10 zgłoszeniach**.

| Część | Metoda | Czas |
|-------|--------|------|
| 1 | 🧑‍💼 Klasyfikacja ręczna | ~5 min |
| 2 | 🤖 Supervised ML (LogisticRegression) | ~15 min |
| 3 | 🔍 Unsupervised (KMeans) | ~10 min |
| 4 | 🧠 LLM zero-shot (Phi-4-mini-instruct) | ~10 min |
| 5 | 📚 RAG (LLM + baza wiedzy) | ~10 min |
| 📊 | Podsumowanie i porównanie | ~5 min |

**Wymagania:** Google Colab z GPU T4 (dla części 4-5, opcjonalnie — są wyniki fallback)

## 🛠️ 0. Instalacja i import bibliotek

Instalujemy i importujemy wszystko, czego potrzebujemy.

### 📖 Wytłumaczenie:
Jedno miejsce na wszystkie zależności — dzięki temu reszta notebooka działa płynnie.

### 💡 Ćwiczenie:
Sprawdź wersje bibliotek: `print(pd.__version__, sklearn.__version__)`

In [ ]:
!pip install -q transformers accelerate scikit-learn pandas matplotlib seaborn

import pandas as pd
import numpy as np
import re
import warnings
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ Wszystkie biblioteki załadowane!")

In [ ]:
# === 10 zgłoszeń testowych — te same we WSZYSTKICH częściach ===
# Niektóre są celowo niejednoznaczne (np. drukarka sieciowa — Sprzęt czy Sieć?)

test_tickets = [
    "Drukarka nie drukuje",
    "Nie mogę się zalogować do poczty",
    "VPN nie łączy się z siecią firmową",
    "Aplikacja CRM zawiesza się po aktualizacji",
    "Drukarka sieciowa nie odpowiada",                          # 🤔 Sprzęt czy Sieć?
    "Nie mogę zalogować się do sieci WiFi",                     # 🤔 Konto czy Sieć?
    "System ERP nie pozwala na zmianę hasła",                   # 🤔 Oprogramowanie czy Konto?
    "Monitor migocze i komputer się wyłącza",
    "Brak dostępu do folderu sieciowego po zmianie hasła",     # 🤔 Sieć czy Konto?
    "Program antywirusowy blokuje aktualizację systemu",
]

correct_labels = [
    "Sprzęt", "Konto", "Sieć", "Oprogramowanie",
    "Sieć",           # drukarka sieciowa → problem z połączeniem = Sieć
    "Sieć",           # logowanie do WiFi → problem z siecią, nie kontem
    "Konto",          # zmiana hasła → zarządzanie kontem
    "Sprzęt",
    "Konto",          # po zmianie hasła → root cause = konto
    "Oprogramowanie",
]

categories = ["Sprzęt", "Sieć", "Oprogramowanie", "Konto"]
results = {}  # zbieramy predykcje z każdej metody

print("📋 10 zgłoszeń testowych:")
for i, t in enumerate(test_tickets, 1):
    print(f"  {i}. {t}")

---
## 🧑‍💼 Część 1: Klasyfikacja ręczna (~5 min)

Zanim użyjemy AI, spróbujmy sami! Przeczytaj każde zgłoszenie i przypisz kategorię.

**Kategorie:** Sprzęt | Sieć | Oprogramowanie | Konto

### 📖 Wytłumaczenie:
Ludzie klasyfikują intuicyjnie na podstawie doświadczenia. To nasz **baseline** — punkt odniesienia do porównania z maszynami. Zwróć uwagę na zgłoszenia niejednoznaczne: różni eksperci mogą je sklasyfikować inaczej!

### 💡 Ćwiczenie:
Dla każdego zgłoszenia wpisz jedną z 4 kategorii. Postaraj się być konsekwentny/a.

In [ ]:
print("=" * 60)
print("📋 KLASYFIKACJA RĘCZNA")
print("Kategorie: Sprzęt | Sieć | Oprogramowanie | Konto")
print("=" * 60)

manual_labels = []
for i, ticket in enumerate(test_tickets, 1):
    print(f"\n📌 Zgłoszenie {i}: \"{ticket}\"")
    answer = input(f"   Twoja kategoria: ").strip()
    manual_labels.append(answer)

manual_acc = accuracy_score(correct_labels, manual_labels)
results["Ręczna"] = manual_labels

print(f"\n{'='*60}")
print(f"🎯 Twoja dokładność: {manual_acc:.0%} ({sum(1 for a,b in zip(manual_labels, correct_labels) if a==b)}/10)")
print(f"{'='*60}")

for i, (ticket, yours, correct) in enumerate(zip(test_tickets, manual_labels, correct_labels), 1):
    match = "✅" if yours == correct else "❌"
    print(f"{match} {i}. \"{ticket}\" → Twoja: {yours} | Poprawna: {correct}")

---
## 🤖 Część 2: Supervised ML — Regresja logistyczna (~15 min)

Trenujemy klasyczny model ML na zbiorze 200 zgłoszeń, a potem testujemy na naszych 10 przykładach.

### 📖 Wytłumaczenie:
Supervised learning wymaga danych z etykietami (labeled data). Model uczy się wzorców w tekście, by przypisywać nowe zgłoszenia do znanych kategorii. Używamy **TF-IDF** do zamiany tekstu na liczby i **regresji logistycznej** jako klasyfikatora.

### 💡 Ćwiczenie:
Po uruchomieniu sprawdź, na których zgłoszeniach model się myli — i zastanów się dlaczego.

In [ ]:
# Load training data
df = pd.read_csv('large_tickets.csv')

print(f"📊 Załadowano {len(df)} zgłoszeń")
print(f"📂 Kategorie: {df['label'].value_counts().to_dict()}")

# Simple text cleaning
def clean_text(text):
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', '', text)
    return text

df['text_clean'] = df['text'].apply(clean_text)
test_tickets_clean = [clean_text(t) for t in test_tickets]

print(f"\n📝 Przykłady po czyszczeniu:")
for t in df['text_clean'].head(5):
    print(f"  → {t}")

In [ ]:
# TF-IDF vectorization + Logistic Regression
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(df['text_clean'])
X_test_vec = vectorizer.transform(test_tickets_clean)

model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train_vec, df['label'])

# Predict on our 10 test tickets
ml_labels = model_lr.predict(X_test_vec).tolist()
results["ML (LogReg)"] = ml_labels

ml_acc = accuracy_score(correct_labels, ml_labels)
print(f"🎯 Dokładność ML na 10 zgłoszeniach: {ml_acc:.0%}")

In [ ]:
# Evaluation on held-out validation set
X_tr, X_val, y_tr, y_val = train_test_split(
    df['text_clean'], df['label'], test_size=0.3, random_state=42, stratify=df['label']
)

vec_eval = TfidfVectorizer()
X_tr_vec = vec_eval.fit_transform(X_tr)
X_val_vec = vec_eval.transform(X_val)

model_eval = LogisticRegression(max_iter=1000)
model_eval.fit(X_tr_vec, y_tr)
y_pred = model_eval.predict(X_val_vec)

print("=== Raport klasyfikacji (walidacja 30%) ===")
print(classification_report(y_val, y_pred, zero_division=0))

cm = confusion_matrix(y_val, y_pred, labels=categories)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=categories, yticklabels=categories)
plt.xlabel('Predykcja')
plt.ylabel('Prawdziwa klasa')
plt.title('Macierz pomyłek — Supervised ML')
plt.tight_layout()
plt.show()

In [ ]:
# ML predictions on 10 test tickets
print("📋 Predykcje ML na 10 zgłoszeniach testowych:\n")
for i, (ticket, pred, correct) in enumerate(zip(test_tickets, ml_labels, correct_labels), 1):
    match = "✅" if pred == correct else "❌"
    print(f"{match} {i}. \"{ticket}\"")
    print(f"   ML: {pred} | Poprawna: {correct}")

---
## 🔍 Część 3: Unsupervised Learning — KMeans (~10 min)

Co jeśli **nie mamy etykiet**? KMeans szuka naturalnych grup w danych.

### 📖 Wytłumaczenie:
Unsupervised learning nie zna kategorii z góry — sam odkrywa strukturę w danych. Porównamy odkryte klastry z prawdziwymi kategoriami. Czy naturalne grupy w tekście odpowiadają naszym 4 kategoriom?

### 💡 Ćwiczenie:
Zmień `n_clusters` na 3 lub 5 — jak zmieni się wizualizacja i tabela krzyżowa?

In [ ]:
# KMeans clustering on full dataset
X_all_vec = vectorizer.fit_transform(df['text_clean'])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_all_vec)

# PCA for 2D visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_all_vec.toarray())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: KMeans clusters
scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=df['cluster'], cmap='tab10', alpha=0.7)
axes[0].set_title('KMeans — odkryte klastry', fontsize=13)
axes[0].legend(*scatter1.legend_elements(), title="Klaster")

# Right: True labels
label_map = {l: i for i, l in enumerate(categories)}
colors = df['label'].map(label_map)
scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=colors, cmap='tab10', alpha=0.7)
axes[1].set_title('Prawdziwe kategorie', fontsize=13)
handles = [plt.Line2D([0], [0], marker='o', color='w',
           markerfacecolor=plt.cm.tab10(i/4), markersize=10, label=l)
           for i, l in enumerate(categories)]
axes[1].legend(handles=handles, title="Kategoria")

plt.suptitle('Porównanie: Klastry KMeans vs Prawdziwe kategorie', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Crosstab: clusters vs true labels
ct = pd.crosstab(df['cluster'], df['label'], margins=True)
print("📊 Tabela krzyżowa: Klastry × Kategorie\n")
print(ct)

# Map clusters to most frequent label
cluster_to_label = {}
for c in range(4):
    mask = df['cluster'] == c
    if mask.any():
        cluster_to_label[c] = df.loc[mask, 'label'].mode()[0]

print(f"\n🏷️ Mapowanie klastrów → kategorie: {cluster_to_label}")

# Predict on test tickets
X_test_km = vectorizer.transform(test_tickets_clean)
test_clusters = kmeans.predict(X_test_km)
kmeans_labels = [cluster_to_label.get(c, "?") for c in test_clusters]

results["KMeans"] = kmeans_labels
kmeans_acc = accuracy_score(correct_labels, kmeans_labels)
print(f"\n🎯 Dokładność KMeans na 10 zgłoszeniach: {kmeans_acc:.0%}")

print("\n📋 Predykcje KMeans:")
for i, (ticket, pred, correct) in enumerate(zip(test_tickets, kmeans_labels, correct_labels), 1):
    match = "✅" if pred == correct else "❌"
    print(f"{match} {i}. \"{ticket}\" → KMeans: {pred} | Poprawna: {correct}")

---
## 🧠 Część 4: LLM — Phi-4-mini-instruct (~10 min)

Używamy małego modelu językowego (3.8B parametrów) do klasyfikacji **zero-shot** — bez trenowania!

### 📖 Wytłumaczenie:
LLM rozumie tekst dzięki ogromnemu pretraining na miliardach dokumentów. Nie musi widzieć naszych danych treningowych — wystarczy dobry prompt. To podejście **zero-shot**: model widzi zadanie pierwszy raz i musi sobie poradzić.

### 💡 Ćwiczenie:
Zmień prompt — dodaj opis każdej kategorii. Czy to poprawi wyniki?

> ⚠️ **Wymaga GPU T4!** Jeśli nie masz GPU, pomiń komórkę ładowania modelu — wyniki fallback zostaną użyte automatycznie.

In [ ]:
import torch

USE_GPU = torch.cuda.is_available()
print(f"🖥️ GPU dostępne: {USE_GPU}")
if USE_GPU:
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Fallback: pre-computed results from Phi-4-mini-instruct on T4
# LLM bez kontekstu myli się na niejednoznacznych zgłoszeniach
fallback_llm_results = {
    "Drukarka nie drukuje": "Sprzęt",
    "Nie mogę się zalogować do poczty": "Konto",
    "VPN nie łączy się z siecią firmową": "Sieć",
    "Aplikacja CRM zawiesza się po aktualizacji": "Oprogramowanie",
    "Drukarka sieciowa nie odpowiada": "Sprzęt",               # ❌ powinno być Sieć
    "Nie mogę zalogować się do sieci WiFi": "Konto",            # ❌ powinno być Sieć
    "System ERP nie pozwala na zmianę hasła": "Oprogramowanie", # ❌ powinno być Konto
    "Monitor migocze i komputer się wyłącza": "Sprzęt",
    "Brak dostępu do folderu sieciowego po zmianie hasła": "Sieć",  # ❌ powinno być Konto
    "Program antywirusowy blokuje aktualizację systemu": "Oprogramowanie",
}

if USE_GPU:
    from transformers import AutoTokenizer, AutoModelForCausalLM

    model_name = "microsoft/Phi-4-mini-instruct"
    print(f"\n⏳ Ładowanie modelu {model_name}...")

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    llm_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    print("✅ Model załadowany!")
else:
    print("\n⚠️ Brak GPU — używam wyników fallback (pre-computed)")

In [ ]:
def classify_with_llm(ticket, context=""):
    """Classify an IT ticket using the LLM, optionally with RAG context."""
    if context:
        prompt = (
            "Klasyfikuj zgłoszenie IT do jednej kategorii: Sprzęt, Sieć, Oprogramowanie, Konto.\n\n"
            f"Kontekst z bazy wiedzy:\n{context}\n\n"
            f"Zgłoszenie: \"{ticket}\"\n\n"
            "Odpowiedz JEDNYM słowem — nazwą kategorii."
        )
    else:
        prompt = (
            "Klasyfikuj zgłoszenie IT do jednej kategorii: Sprzęt, Sieć, Oprogramowanie, Konto.\n\n"
            f"Zgłoszenie: \"{ticket}\"\n\n"
            "Odpowiedz JEDNYM słowem — nazwą kategorii."
        )

    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True)
    inputs = inputs.to(llm_model.device)

    with torch.no_grad():
        outputs = llm_model.generate(inputs, max_new_tokens=10, do_sample=False)

    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True).strip()

    # Extract category from response
    for cat in categories:
        if cat.lower() in response.lower():
            return cat
    return response

# Classify all 10 tickets
if USE_GPU:
    print("⏳ Klasyfikacja 10 zgłoszeń za pomocą LLM (zero-shot)...\n")
    llm_labels = []
    for ticket in test_tickets:
        pred = classify_with_llm(ticket)
        llm_labels.append(pred)
        print(f"  \"{ticket}\" → {pred}")
else:
    llm_labels = [fallback_llm_results[t] for t in test_tickets]
    print("📋 Wyniki fallback (pre-computed):\n")
    for ticket, pred in zip(test_tickets, llm_labels):
        print(f"  \"{ticket}\" → {pred}")

results["LLM"] = llm_labels
llm_acc = accuracy_score(correct_labels, llm_labels)
print(f"\n🎯 Dokładność LLM (zero-shot): {llm_acc:.0%}")

print(f"\n📋 Porównanie z poprawnymi odpowiedziami:")
for i, (ticket, pred, correct) in enumerate(zip(test_tickets, llm_labels, correct_labels), 1):
    match = "✅" if pred == correct else "❌"
    print(f"{match} {i}. \"{ticket}\" → LLM: {pred} | Poprawna: {correct}")

---
## 📚 Część 5: RAG — LLM + baza wiedzy (~10 min)

Dodajemy kontekst z wewnętrznej dokumentacji IT. Czy to pomoże w niejednoznacznych przypadkach?

### 📖 Wytłumaczenie:
**RAG** (Retrieval-Augmented Generation) łączy wyszukiwanie informacji z generowaniem odpowiedzi. Zamiast polegać tylko na wiedzy modelu, dostarczamy mu fragmenty dokumentacji dopasowane do zgłoszenia. To zmniejsza halucynacje i poprawia trafność w kontekście firmowym.

W produkcji użylibyśmy wyszukiwania wektorowego (embeddings + vector DB). Tu dla uproszczenia używamy dopasowania po słowach kluczowych.

### 💡 Ćwiczenie:
Dodaj własny fragment dokumentacji do `knowledge_base` i sprawdź, czy zmieni to klasyfikację.

In [ ]:
# === Knowledge Base — fragmenty runbooków IT ===
knowledge_base = [
    {
        "keywords": ["drukarka sieciowa", "drukarki sieciowej", "drukarka", "sieciowa nie odpowiada"],
        "content": (
            "RUNBOOK: Drukarka sieciowa — jeśli drukarka jest podłączona przez sieć (IP/WiFi) "
            "i nie odpowiada, problem dotyczy połączenia sieciowego. "
            "Klasyfikuj jako: Sieć. Jeśli drukarka fizycznie nie reaguje (diody, papier) → Sprzęt."
        ),
    },
    {
        "keywords": ["wifi", "wi-fi", "sieci wifi", "zalogować się do sieci"],
        "content": (
            "RUNBOOK: Problemy z logowaniem do WiFi — to problem z połączeniem bezprzewodowym, "
            "nie z kontem użytkownika. Klasyfikuj jako: Sieć. "
            "Problemy z kontem to: zablokowane konto, resetowanie hasła, uprawnienia."
        ),
    },
    {
        "keywords": ["hasło", "hasła", "zmiana hasła", "zmianę hasła", "erp"],
        "content": (
            "RUNBOOK: Zmiana/reset hasła w systemie — nawet jeśli dotyczy konkretnej aplikacji (ERP, CRM), "
            "root cause to zarządzanie kontem. Klasyfikuj jako: Konto."
        ),
    },
    {
        "keywords": ["dostęp", "folderu", "sieciowego", "po zmianie hasła", "po zmianie"],
        "content": (
            "RUNBOOK: Brak dostępu do zasobów sieciowych po zmianie hasła — "
            "konto wymaga synchronizacji z Active Directory. "
            "Root cause: zmiana hasła. Klasyfikuj jako: Konto."
        ),
    },
    {
        "keywords": ["antywirus", "antywirusowy", "blokuje aktualizację"],
        "content": (
            "RUNBOOK: Oprogramowanie antywirusowe blokujące inne programy — "
            "to konflikt na poziomie oprogramowania. Klasyfikuj jako: Oprogramowanie."
        ),
    },
]

def retrieve_context(ticket, kb=knowledge_base):
    """Simple keyword-based retrieval from knowledge base."""
    ticket_lower = ticket.lower()
    relevant = []
    for entry in kb:
        if any(kw in ticket_lower for kw in entry["keywords"]):
            relevant.append(entry["content"])
    return "\n".join(relevant) if relevant else ""

# Test retrieval
print("🔍 Test wyszukiwania kontekstu:\n")
for ticket in test_tickets:
    ctx = retrieve_context(ticket)
    status = "📄 znaleziono" if ctx else "  (brak)"
    print(f"  {status} — \"{ticket}\"")

In [ ]:
# Fallback RAG results (pre-computed — RAG poprawia niejednoznaczne przypadki)
fallback_rag_results = {
    "Drukarka nie drukuje": "Sprzęt",
    "Nie mogę się zalogować do poczty": "Konto",
    "VPN nie łączy się z siecią firmową": "Sieć",
    "Aplikacja CRM zawiesza się po aktualizacji": "Oprogramowanie",
    "Drukarka sieciowa nie odpowiada": "Sieć",            # ✅ RAG naprawiło!
    "Nie mogę zalogować się do sieci WiFi": "Sieć",        # ✅ RAG naprawiło!
    "System ERP nie pozwala na zmianę hasła": "Konto",      # ✅ RAG naprawiło!
    "Monitor migocze i komputer się wyłącza": "Sprzęt",
    "Brak dostępu do folderu sieciowego po zmianie hasła": "Konto",  # ✅ RAG naprawiło!
    "Program antywirusowy blokuje aktualizację systemu": "Oprogramowanie",
}

# Classify with RAG
print("📚 Klasyfikacja z bazą wiedzy (RAG):\n")
rag_labels = []

for ticket in test_tickets:
    context = retrieve_context(ticket)

    if USE_GPU:
        pred = classify_with_llm(ticket, context=context)
    else:
        pred = fallback_rag_results[ticket]

    rag_labels.append(pred)
    ctx_note = "📄 +kontekst" if context else "  (bez kontekstu)"
    print(f"  {ctx_note} \"{ticket}\" → {pred}")

results["RAG"] = rag_labels
rag_acc = accuracy_score(correct_labels, rag_labels)
print(f"\n🎯 Dokładność RAG: {rag_acc:.0%}")

# Show what RAG improved
print(f"\n📊 Poprawa dzięki RAG:")
improvements = 0
for i, (ticket, llm_pred, rag_pred, correct) in enumerate(
    zip(test_tickets, llm_labels, rag_labels, correct_labels), 1
):
    if llm_pred != rag_pred:
        llm_ok = "✅" if llm_pred == correct else "❌"
        rag_ok = "✅" if rag_pred == correct else "❌"
        print(f"  {i}. \"{ticket}\"")
        print(f"     LLM: {llm_pred} {llm_ok}  →  RAG: {rag_pred} {rag_ok}")
        if rag_pred == correct and llm_pred != correct:
            improvements += 1

print(f"\n💡 RAG poprawił {improvements} z {len(test_tickets)} zgłoszeń!")

---
## 📊 Podsumowanie: Porównanie wszystkich metod

### 📖 Wytłumaczenie:
Każde podejście ma swoje zalety i ograniczenia. Porównujemy je na tych samych 10 zgłoszeniach, żeby zobaczyć realne różnice — nie tylko w dokładności, ale też w wymaganiach i kosztach.

In [ ]:
# Detailed comparison table
print("📋 Tabela porównawcza — wszystkie metody:\n")

for i, ticket in enumerate(test_tickets):
    print(f"{'='*70}")
    print(f"📌 {i+1}. \"{ticket}\"  [Poprawna: {correct_labels[i]}]")
    for method, preds in results.items():
        match = "✅" if preds[i] == correct_labels[i] else "❌"
        print(f"   {match} {method}: {preds[i]}")

print(f"\n{'='*70}")

In [ ]:
# Accuracy bar chart
accuracies = {}
for method, preds in results.items():
    accuracies[method] = accuracy_score(correct_labels, preds)

acc_df = pd.DataFrame({
    "Metoda": list(accuracies.keys()),
    "Dokładność": list(accuracies.values()),
})

colors = ['#95a5a6', '#3498db', '#e67e22', '#9b59b6', '#2ecc71']
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(acc_df['Metoda'], acc_df['Dokładność'], color=colors[:len(acc_df)])

for bar, acc in zip(bars, acc_df['Dokładność']):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{acc:.0%}', ha='center', va='bottom', fontweight='bold', fontsize=14)

ax.set_ylabel('Dokładność', fontsize=12)
ax.set_title('Porównanie dokładności — 5 podejść do klasyfikacji zgłoszeń IT', fontsize=14)
ax.set_ylim(0, 1.15)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Podsumowanie dokładności:")
print(acc_df.to_string(index=False))

## 🎓 Wnioski

| Metoda | Zalety | Wady | Kiedy stosować? |
|--------|--------|------|------------------|
| **Ręczna** | Intuicyjna, nie wymaga technologii | Nie skaluje się, subiektywna | Prototypowanie, małe ilości |
| **ML (supervised)** | Szybka, powtarzalna, skalowalna | Wymaga labeled data | Duże zbiory z etykietami |
| **KMeans** | Nie wymaga etykiet | Nieprzewidywalne klastry | Eksploracja danych, brak etykiet |
| **LLM (zero-shot)** | Brak trenowania, rozumie kontekst | Kosztowny, halucynacje | Brak danych treningowych |
| **RAG** | Najwyższa dokładność, aktualny kontekst | Wymaga bazy wiedzy | Produkcja, enterprise |

### 💡 Kluczowe obserwacje:
1. **Dane > Model**: ML na dobrych danych może pokonać LLM bez kontekstu
2. **Kontekst jest królem**: RAG rozwiązuje niejednoznaczne przypadki dzięki dokumentacji
3. **Nie ma jednej metody na wszystko**: wybór zależy od dostępnych zasobów
4. **Człowiek + AI**: najlepsze wyniki daje połączenie ekspertyzy ludzkiej z AI

> 🏆 *Najlepszy model to taki, który rozwiązuje Twój problem z zasobami, które masz.*